# Maintain_Delta_Tables

Routine storage maintenance for the ColdTrace Medallion Lakehouse: `OPTIMIZE ... ZORDER BY` compacts small files on the large, frequently joined tables; `VACUUM ... RETAIN 168 HOURS` removes old file versions past the 7-day safety window.

Runs daily in its own pipeline (`PL_ColdTrace_Storage_Maintenance`), separate from the hourly `PL_ColdTrace_Master`. See deployment notes below.

In [1]:
from datetime import datetime

# Tables that benefit most from OPTIMIZE ZORDER, with the columns most
# commonly used to filter or join against them.
tables_to_optimize = {
    "silver_orders": ["CustomerId"],
    "silver_vehicle_movements": ["RouteId"],
    "gold.fact_sales": ["CustomerId", "ProductId", "OrderDate"],
    "gold.fact_vehicle_movements": ["RouteId", "EventDate"],
}

# All tables to VACUUM, including small dimension/audit tables that don't
# need ZORDER but still accumulate old file versions over time.
all_tables = list(tables_to_optimize.keys()) + [
    "gold.dim_customer",
    "gold.dim_date",
    "gold.dim_product",
    "gold.dim_route",
    "silver_pipeline_audit",
]

VACUUM_RETENTION_HOURS = 168  # Delta default: 7-day safety window

StatementMeta(, c92ab78c-d555-4cbb-ae50-58f1d4013128, 3, Finished, Available, Finished, False)

In [2]:
# Step 1: OPTIMIZE (Z-Order) the large, frequently joined tables.
print("=" * 60)
print(f" STORAGE MAINTENANCE RUN — {datetime.now()}")
print("=" * 60)

optimize_failures = []

for table_name, zorder_cols in tables_to_optimize.items():
    if not spark.catalog.tableExists(table_name):
        print(f"-> {table_name}: does not exist, skipped")
        continue
    cols = ", ".join(zorder_cols)
    print(f"-> OPTIMIZE {table_name} ZORDER BY ({cols}) ...")
    try:
        spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({cols})")
    except Exception as e:
        print(f"   FAILED: {e}")
        optimize_failures.append(table_name)

StatementMeta(, c92ab78c-d555-4cbb-ae50-58f1d4013128, 4, Finished, Available, Finished, False)

 STORAGE MAINTENANCE RUN — 2026-09-23 08:14:57.678791
-> OPTIMIZE silver_orders ZORDER BY (CustomerId) ...
-> OPTIMIZE silver_vehicle_movements ZORDER BY (RouteId) ...
-> OPTIMIZE gold.fact_sales ZORDER BY (CustomerId, ProductId, OrderDate) ...
-> OPTIMIZE gold.fact_vehicle_movements ZORDER BY (RouteId, EventDate) ...


In [3]:
# Step 2: VACUUM stale file versions on every relevant table.
vacuum_failures = []

for table_name in all_tables:
    if not spark.catalog.tableExists(table_name):
        continue
    print(f"-> VACUUM {table_name} RETAIN {VACUUM_RETENTION_HOURS} HOURS ...")
    try:
        spark.sql(f"VACUUM {table_name} RETAIN {VACUUM_RETENTION_HOURS} HOURS")
    except Exception as e:
        print(f"   FAILED: {e}")
        vacuum_failures.append(table_name)

print("=" * 60)
if optimize_failures or vacuum_failures:
    print(f" Maintenance run completed WITH FAILURES — OPTIMIZE: {optimize_failures or 'none'}, VACUUM: {vacuum_failures or 'none'}")
else:
    print(" Maintenance run complete — no failures.")
print("=" * 60)

StatementMeta(, c92ab78c-d555-4cbb-ae50-58f1d4013128, 5, Finished, Available, Finished, False)

-> VACUUM silver_orders RETAIN 168 HOURS ...
-> VACUUM silver_vehicle_movements RETAIN 168 HOURS ...
-> VACUUM gold.fact_sales RETAIN 168 HOURS ...
-> VACUUM gold.fact_vehicle_movements RETAIN 168 HOURS ...
-> VACUUM gold.dim_customer RETAIN 168 HOURS ...
-> VACUUM gold.dim_date RETAIN 168 HOURS ...
-> VACUUM gold.dim_product RETAIN 168 HOURS ...
-> VACUUM gold.dim_route RETAIN 168 HOURS ...
-> VACUUM silver_pipeline_audit RETAIN 168 HOURS ...
 Maintenance run complete — no failures.


## Deployment notes

Lakehouse: `LH_ColdTrace_Medallion` in `ColdTrace-Analytics` (same as every other Silver/Gold notebook). Pipeline: its own (`PL_ColdTrace_Storage_Maintenance`), separate from `PL_ColdTrace_Master` — chaining it hourly would trigger OPTIMIZE far more often than needed. Trigger: weekly Schedule, off-peak (e.g. Sunday 02:00).

VACUUM's 168-hour (7-day) retention lines up neatly with the weekly cadence — each run reclaims exactly the space freed since the previous one, no stale file versions ever pile up in between.